In [4]:
!pip install -q transformers datasets trl accelerate



In [5]:
from google.colab import files
uploaded = files.upload()  # select "data.json"


Saving data(1).json to data(1) (1).json


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="data(1).json",
    split="train"
)

print("Total examples:", len(dataset))
print("First example:\n", dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Total examples: 10
First example:
 {'id': 'gsm8k_001', 'question': 'Natalia sold 48 clips in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'cot': 'Step 1: Natalia sold 48 clips in April. Step 2: In May, she sold half as many clips as in April, which is 24 clips. Step 3: The total number of clips sold in April and May is 48 plus 24, which equals 72.', 'final_answer': '72'}


In [8]:
def format_example(e):
    text = f"""### Question:
{e['question']}

### Reasoning:
{e['cot']}

### Answer:
{e['final_answer']}
"""
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### Question:
Natalia sold 48 clips in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

### Reasoning:
Step 1: Natalia sold 48 clips in April. Step 2: In May, she sold half as many clips as in April, which is 24 clips. Step 3: The total number of clips sold in April and May is 48 plus 24, which equals 72.

### Answer:
72



In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # small + fast
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    max_steps=100
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

In [16]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.3313296508789063, metrics={'train_runtime': 22.7903, 'train_samples_per_second': 17.551, 'train_steps_per_second': 4.388, 'total_flos': 17614054785024.0, 'train_loss': 1.3313296508789063})

In [17]:
prompt = """### Question:
If x + 5 = 12, find x.

### Reasoning:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Question:
If x + 5 = 12, find x.

### Reasoning:
Step 1: In an integer, x < y , which has the same value as y. So, y = 2^3. Step 2: In the same constant expression, y < 3 = 4^6. Step 3: Now evaluate the identity of all powers, and compare them by their constant identity: 5(a^2 + 2^3) = 3^3. Step 4: Substitute the exponentiation constants: 8/3 = 3^3. Step 5: We


In [18]:
trainer.model.save_pretrained("./cot-sft-demo-model")
tokenizer.save_pretrained("./cot-sft-demo-model")

!zip -r cot-sft-demo-model.zip ./cot-sft-demo-model

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: cot-sft-demo-model/ (stored 0%)
  adding: cot-sft-demo-model/config.json (deflated 53%)
  adding: cot-sft-demo-model/tokenizer.json (deflated 82%)
  adding: cot-sft-demo-model/generation_config.json (deflated 23%)
  adding: cot-sft-demo-model/tokenizer_config.json (deflated 48%)
  adding: cot-sft-demo-model/model.safetensors (deflated 7%)


In [19]:
!ls -lh

total 291M
drwxr-xr-x 2 root root 4.0K Mar  1 06:35  cot-sft-demo-model
-rw-r--r-- 1 root root 291M Mar  1 06:36  cot-sft-demo-model.zip
-rw-r--r-- 1 root root 5.5K Mar  1 06:19 'data(1) (1).json'
-rw-r--r-- 1 root root 5.5K Mar  1 05:37 'data(1).json'
drwxr-xr-x 3 root root 4.0K Mar  1 06:28  results
drwxr-xr-x 1 root root 4.0K Feb  6 14:31  sample_data


In [20]:
from google.colab import files
files.download("cot-sft-demo-model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>